# API Basics Practice Exercises (Notebook)

**Estimated time:** ~8 hours (part of the ~12 hour basics level, alongside `api-basics-guide.ipynb`).

These exercises match the topics in the guide. Each exercise has `assert` checks — when a cell prints `OK`, you got it right. Replace `# Your code here` with your solution; keep the asserts as they are.

Run this first:

In [ ]:
from fastapi import FastAPI, HTTPException
from fastapi.testclient import TestClient
from pydantic import BaseModel

print("Imports OK")

## Exercise 1: JSON Warm-up

1. Convert the `movie` dict to a JSON string called `movie_json`.
2. Parse `incoming` (a JSON string) into a dict called `order`.
3. From `order`, compute `total` = price × quantity.

In [ ]:
import json

movie = {"title": "Inception", "year": 2010, "genres": ["sci-fi", "thriller"]}
incoming = '{"item": "keyboard", "price": 1500, "quantity": 2}'

# Your code here

assert isinstance(movie_json, str) and json.loads(movie_json) == movie
assert order["item"] == "keyboard"
assert total == 3000
print("OK")

## Exercise 2: Hello, API

Build a FastAPI app with one endpoint: `GET /` returning `{"message": "pong"}`.

In [ ]:
# Your code here

client = TestClient(app)
response = client.get("/")
assert response.status_code == 200
assert response.json() == {"message": "pong"}
print("OK")

## Exercise 3: Path Parameters

Build an app with:

1. `GET /square/{n}` — `n` is an int; return `{"result": n squared}`.
2. `GET /shout/{word}` — `word` is a string; return `{"result": word in uppercase}`.

In [ ]:
# Your code here

client = TestClient(app)
assert client.get("/square/7").json() == {"result": 49}
assert client.get("/shout/hello").json() == {"result": "HELLO"}
assert client.get("/square/oops").status_code == 422  # type validation is automatic
print("OK")

## Exercise 4: Query Parameters

Build `GET /greet` with:

- required query parameter `name` (str)
- optional `lang` (str, default `"en"`)

Return `{"greeting": "Hello, <name>!"}` for `en` and `{"greeting": "Bonjour, <name>!"}` for `fr`.

In [ ]:
# Your code here

client = TestClient(app)
assert client.get("/greet?name=Asha").json() == {"greeting": "Hello, Asha!"}
assert client.get("/greet?name=Asha&lang=fr").json() == {"greeting": "Bonjour, Asha!"}
assert client.get("/greet").status_code == 422  # name is required
print("OK")

## Exercise 5: Request Body with Pydantic

1. Define a Pydantic model `Product` with fields: `name` (str), `price` (float), `in_stock` (bool, default `True`).
2. Build `POST /products` that accepts a `Product` and returns `{"name": ..., "price_with_tax": price * 1.18}`.

In [ ]:
# Your code here

client = TestClient(app)
r = client.post("/products", json={"name": "Mouse", "price": 500})
assert r.status_code == 200
assert r.json() == {"name": "Mouse", "price_with_tax": 590.0}
assert client.post("/products", json={"name": "Mouse"}).status_code == 422  # price missing
print("OK")

## Exercise 6: Status Codes and Errors

Given the `CITIES` dict, build:

1. `GET /cities/{code}` — return `{"code": ..., "city": ...}`, or a **404** with detail `"Unknown city code"` if the code is not in `CITIES`.
2. `POST /cities` — accepts a JSON body (use a `dict` or a model), returns it back, and responds with status **201**.

In [ ]:
CITIES = {"BLR": "Bengaluru", "DEL": "Delhi", "MAA": "Chennai"}

# Your code here

client = TestClient(app)
assert client.get("/cities/BLR").json() == {"code": "BLR", "city": "Bengaluru"}
missing = client.get("/cities/XYZ")
assert missing.status_code == 404
assert missing.json()["detail"] == "Unknown city code"
assert client.post("/cities", json={"code": "HYD", "city": "Hyderabad"}).status_code == 201
print("OK")

## Exercise 7: Mini Project — Books CRUD API

Build a complete in-memory books API:

- `POST /books` (status 201) — body has `title` (str) and `author` (str); assign incremental integer ids starting at 1; return `{"id": ..., "title": ..., "author": ...}`
- `GET /books` — list all books (same shape, as a list)
- `GET /books/{book_id}` — one book, or 404
- `DELETE /books/{book_id}` (status 204) — remove a book, or 404

Use a Pydantic model for the body and a dict as storage, like the guide's todo example.

In [ ]:
# Your code here

client = TestClient(app)

r1 = client.post("/books", json={"title": "Deep Work", "author": "Cal Newport"})
r2 = client.post("/books", json={"title": "Atomic Habits", "author": "James Clear"})
assert r1.status_code == 201 and r1.json()["id"] == 1
assert r2.json()["id"] == 2

assert len(client.get("/books").json()) == 2
assert client.get("/books/1").json()["title"] == "Deep Work"
assert client.get("/books/99").status_code == 404

assert client.delete("/books/1").status_code == 204
assert client.get("/books/1").status_code == 404
assert len(client.get("/books").json()) == 1
print("OK")

## Exercise 8: Run It for Real (no asserts)

1. Copy your books API from Exercise 7 into a new file `main.py`.
2. In a terminal run: `uvicorn main:app --reload`
3. Open `http://localhost:8000/docs` in your browser.
4. Use the Swagger UI to create two books, list them, and delete one — all from the browser.
5. Stop the server with `Ctrl+C`.

This is exactly how FastAPI apps run in production (minus `--reload`).

## Done!

If every cell printed `OK`, you have the fundamentals of building web APIs. Continue with `api-intermediate-guide.ipynb` for validation in depth, routers, dependency injection, authentication, middleware, databases, and testing.